In [ ]:
from snowflake.snowpark.functions import col, year, avg
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

session = get_active_session()



In [ ]:
market = session.table("MARKET_RISK_VALUE")
weather = session.table("WEATHER_RISK_DATA")
land = session.table("LAND_VALUE_DATA")

print("Market rows:", market.count())
print("Weather rows:", weather.count())
print("Land rows:", land.count())


In [ ]:
from snowflake.snowpark.functions import avg, when, col

market_features = (
    market
    .group_by("REGION", "YEAR")
    .agg(
        avg(when(col("COMMODITY") == "Corn", col("PRICE_USD_PER_BUSHEL"))).alias("CORN_PRICE"),
        avg(when(col("COMMODITY") == "Soybeans", col("PRICE_USD_PER_BUSHEL"))).alias("SOYBEAN_PRICE"),
        avg(when(col("COMMODITY") == "Hard Red Winter Wheat", col("PRICE_USD_PER_BUSHEL"))).alias("WHEAT_PRICE")
    )
)

print("Market features", market_features.count())
market_features.show(10)

In [ ]:
from snowflake.snowpark.functions import avg
weather_features = (
    weather
    .group_by("REGION","YEAR")
    .agg(
        avg("PRECIPITATION_INCHES").alias("PRECIP"),
        avg("AVERAGE_TEMPERATURE").alias("TEMP"),
        avg("PALMER_DROUGHT_SEVERITY_INDEX").alias("DROUGHT"),
        avg("HEATING_DEGREE_DAYS").alias("HEAT"),
        avg("COOLING_DEGREE_DAYS").alias("COOL")
    )
)

print("Weather features", weather_features.count())
weather_features.show(10)

In [ ]:
from snowflake.snowpark.functions import when, lit, col
land_features = (
    land.select(

        "REGION",

        "YEAR",



        col("LAND_VALUE_AVG_PER_ACRE").alias("LAND_VALUE_AVG"),

        col("ALL_CROPLAND_PER_ACRE").alias("CROPLAND_VALUE"),

        col("FARM_REAL_ESTATE_PER_ACRE").alias("FARM_RE_VALUE"),



        col("CASH_RENT_DRYLAND_PER_ACRE").alias("RENT_DRYLAND"),

        col("CASH_RENT_IRRIGATED_PER_ACRE").alias("RENT_IRRIGATED"),

        col("CASH_RENT_PASTURE_PER_ACRE").alias("RENT_PASTURE"),



        col("LAND_VALUE_HRWW_PER_ACRE").alias("LAND_WHEAT"),

        col("LAND_VALUE_CORN_PER_ACRE").alias("LAND_CORN"),

        col("LAND_VALUE_SOYBEANS_PER_ACRE").alias("LAND_SOYBEANS"),



        col("AVG_YOY_ABS_CHG").alias("AVG_YOY_ABS"),

        col("AVG_YOY_PCT_CHG").alias("AVG_YOY_PCT"),



        col("CORN_YOY_ABS_CHG").alias("CORN_YOY_ABS"),

        col("CORN_YOY_PCT_CHG").alias("CORN_YOY_PCT"),



        col("SOYBEANS_YOY_ABS_CHG").alias("SOY_YOY_ABS"),

        col("SOYBEANS_YOY_PCT_CHG").alias("SOY_YOY_PCT"),



        col("HRWW_YOY_ABS_CHG").alias("WHEAT_YOY_ABS"),

        col("HRWW_YOY_PCT_CHG").alias("WHEAT_YOY_PCT")

    )

)



print("Land features", land_features.count())

land_features.show(10)

In [ ]:
from snowflake.snowpark.functions import when, col, lit

land_features = land_features.with_column(
"REGION",
when(col("REGION") == "northwest", lit("Northwest Kansas"))
.when(col("REGION") == "central", lit("Central Kansas"))
.when(col("REGION") == "southeast", lit("Southeast Kansas"))
.when(col("REGION") == "southwest", lit("Southwest Kansas"))
.when(col("REGION") == "northeast", lit("Northeast Kansas"))
.otherwise(col("REGION"))
)

print("Land regions after fix:")
land_features.select("REGION").distinct().show()




In [ ]:
features = (
    weather_features
    .join(land_features, ["REGION", "YEAR"])
    .join(market_features, ["REGION", "YEAR"])
)

print("Joined dataset rows:", features.count())
features.show(10)

In [ ]:
market_state = (

    market

    .group_by("YEAR", "REGION")

    .agg(

        avg(when(col("COMMODITY") == "Corn", col("PRICE_USD_PER_BUSHEL"))).alias("CORN_PRICE"),

        avg(when(col("COMMODITY") == "Soybeans", col("PRICE_USD_PER_BUSHEL"))).alias("SOYBEAN_PRICE"),

        avg(when(col("COMMODITY") == "Hard Red Winter Wheat", col("PRICE_USD_PER_BUSHEL"))).alias("WHEAT_PRICE")

    )

)



features = (
    weather_features
    .join(land_features, ["REGION", "YEAR"])
    .join(market_features, ["REGION", "YEAR"])
)

print("Joined rows after REGION+YEAR market join:", features.count())
features.show(20)

In [ ]:
df = features.to_pandas()

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())

In [ ]:
import pandas as pd



numeric_cols = [

    "CORN_PRICE",

    "SOYBEAN_PRICE",

    "WHEAT_PRICE",

    "PRECIP",

    "TEMP",

    "DROUGHT",

    "HEAT",

    "COOL",

    "LAND_VALUE_AVG",

    "CROPLAND_VALUE",

    "FARM_RE_VALUE",

    "RENT_DRYLAND",

    "RENT_IRRIGATED",

    "RENT_PASTURE",

    "LAND_WHEAT",

    "LAND_CORN",

    "LAND_SOYBEANS",

    "AVG_YOY_ABS",

    "AVG_YOY_PCT",

    "CORN_YOY_ABS",

    "CORN_YOY_PCT",

    "SOY_YOY_ABS",

    "SOY_YOY_PCT",

    "WHEAT_YOY_ABS",

    "WHEAT_YOY_PCT"

]



for c in numeric_cols:

    df[c] = pd.to_numeric(df[c], errors="coerce")



df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())



print("Null counts after fill:")

print(df[numeric_cols].isnull().sum())



In [ ]:
df["MARKET_VALUE"] = (

    df["CORN_PRICE"] +

    df["SOYBEAN_PRICE"] +

    df["WHEAT_PRICE"]

) / 3

In [ ]:
df["TARGET_RISK"] = (

    0.40 * df["MARKET_VALUE"] +

    0.35 * df["DROUGHT"] +

    0.25 * df["LAND_VALUE_AVG"]

)



df["TARGET_RISK"] = 99 * (

    (df["TARGET_RISK"] - df["TARGET_RISK"].min()) /

    (df["TARGET_RISK"].max() - df["TARGET_RISK"].min())

)

In [ ]:
feature_cols = [

    "CORN_PRICE",

    "SOYBEAN_PRICE",

    "WHEAT_PRICE",

    "PRECIP",

    "TEMP",

    "DROUGHT",

    "HEAT",

    "COOL",

    "LAND_VALUE_AVG",

    "CROPLAND_VALUE",

    "FARM_RE_VALUE",

    "RENT_DRYLAND",

    "RENT_IRRIGATED",

    "RENT_PASTURE",

    "LAND_WHEAT",

    "LAND_CORN",

    "LAND_SOYBEANS",

    "AVG_YOY_ABS",

    "AVG_YOY_PCT",

    "CORN_YOY_ABS",

    "CORN_YOY_PCT",

    "SOY_YOY_ABS",

    "SOY_YOY_PCT",

    "WHEAT_YOY_ABS",

    "WHEAT_YOY_PCT"

]



X = df[feature_cols]

y = df["TARGET_RISK"]

In [ ]:
print("market_features rows:", market_features.count())
print("weather_features rows:", weather_features.count())
print("land_features rows:", land_features.count())

In [ ]:
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split

from sklearn.metrics import r2_score, mean_squared_error



X_train, X_test, y_train, y_test = train_test_split(

    X, y, test_size=0.2, random_state=42

)



model = RandomForestRegressor(

    n_estimators=300,

    max_depth=10,

    random_state=42

)



model.fit(X_train, y_train)



preds = model.predict(X_test)



r2 = r2_score(y_test, preds)

rmse = mean_squared_error(y_test, preds, squared=False)



print("Model R2:", r2)

print("RMSE:", rmse)

In [ ]:
preds = model.predict(X_test)

r2 = r2_score(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)

print("Model R2:", r2)
print("RMSE:", rmse)

In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

In [ ]:
df["RISK_SCORE"] = model.predict(X)

df["RISK_SCORE"] = df["RISK_SCORE"].clip(0,99).round().astype(int)

In [ ]:
def classify(score):

    if score <= 19:
        return "Very Low"

    elif score <= 39:
        return "Low"

    elif score <= 59:
        return "Medium"

    elif score <= 79:
        return "High"

    else:
        return "Very High"

df["RISK_LEVEL"] = df["RISK_SCORE"].apply(classify)

In [ ]:
df["AVG_MARKET_PRICE"] = (
    df["CORN_PRICE"] + df["SOYBEAN_PRICE"] + df["WHEAT_PRICE"]
) / 3.0
def get_risk_description(row):
    score = row["RISK_SCORE"]

    # Base level text
    if score <= 20:
        level_text = "Very low risk"
    elif score <= 40:
        level_text = "Low risk"
    elif score <= 60:
        level_text = "Moderate risk"
    elif score <= 80:
        level_text = "High risk"
    else:
        level_text = "Very high risk"

    reasons = []

    # Market condition
    if row["AVG_MARKET_PRICE"] >= df["AVG_MARKET_PRICE"].median():
        reasons.append("market prices are relatively strong")
    else:
        reasons.append("market prices are relatively weak")

    # Precipitation
    if row["PRECIP"] >= df["PRECIP"].median():
        reasons.append("precipitation levels are more favorable")
    else:
        reasons.append("precipitation levels are less favorable")

    # Temperature
    if row["TEMP"] >= df["TEMP"].median():
        reasons.append("temperature levels are higher and may increase crop stress")
    else:
        reasons.append("temperature levels are more moderate")

    # Drought
    if row["DROUGHT"] >= df["DROUGHT"].median():
        reasons.append("drought conditions are elevated")
    else:
        reasons.append("drought pressure is lower")

    # Land value
    if row["LAND_VALUE_AVG"] >= df["LAND_VALUE_AVG"].median():
        reasons.append("land values are higher, which may increase financial pressure")
    else:
        reasons.append("land values are more manageable")

    return f"{level_text} - " + ", ".join(reasons) + "."

df["RISK_DESCRIPTION"] = df.apply(get_risk_description, axis=1)

print(df[[
    "REGION",
    "YEAR",
    "RISK_SCORE",
    "RISK_LEVEL",
    "RISK_DESCRIPTION"
]].head(20))

risk_scores_df = df[[
    "REGION",
    "YEAR",
    "RISK_SCORE",
    "RISK_LEVEL",
    "RISK_DESCRIPTION"
]].reset_index(drop=True)

session.write_pandas(
    risk_scores_df,
    "RISK_SCORES",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------
# CHOOSE YEAR TO RANK
# ---------------------------------------------
rank_year = 2026   # change to 2025 if you want that year instead

rank_df = final_future[final_future["YEAR"] == rank_year].copy()

# ---------------------------------------------
# SAFETY CHECK
# ---------------------------------------------
required_cols = [
    "REGION",
    "YEAR",
    "PREDICTED_RISK_SCORE",
    "DROUGHT",
    "PRECIP",
    "TEMP",
    "LAND_VALUE_AVG",
    "CORN_PRICE",
    "SOYBEAN_PRICE",
    "WHEAT_PRICE"
]

missing_cols = [col for col in required_cols if col not in rank_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# ---------------------------------------------
# CREATE HELPER BENCHMARKS
# ---------------------------------------------
avg_drought = rank_df["DROUGHT"].mean()
avg_precip = rank_df["PRECIP"].mean()
avg_temp = rank_df["TEMP"].mean()
avg_land = rank_df["LAND_VALUE_AVG"].mean()
avg_corn = rank_df["CORN_PRICE"].mean()
avg_soy = rank_df["SOYBEAN_PRICE"].mean()
avg_wheat = rank_df["WHEAT_PRICE"].mean()
avg_risk = rank_df["PREDICTED_RISK_SCORE"].mean()

rank_df["AVG_MARKET_PRICE"] = (
    rank_df["CORN_PRICE"] +
    rank_df["SOYBEAN_PRICE"] +
    rank_df["WHEAT_PRICE"]
) / 3.0
avg_market_price = rank_df["AVG_MARKET_PRICE"].mean()

# ---------------------------------------------
# ADVANCED RISK LEVEL FUNCTION
# ---------------------------------------------
def classify_risk(score):
    if score <= 19:
        return "Very Low"
    elif score <= 39:
        return "Low"
    elif score <= 59:
        return "Moderate"
    elif score <= 79:
        return "High"
    else:
        return "Very High"

# ---------------------------------------------
# SORT AND RANK FIRST
# ---------------------------------------------
rank_df = rank_df.sort_values(
    by=["PREDICTED_RISK_SCORE", "DROUGHT", "LAND_VALUE_AVG"],
    ascending=[True, True, True]
).reset_index(drop=True)

rank_df["REGION_RANK"] = range(1, len(rank_df) + 1)
total_regions = len(rank_df)

rank_df["RANK_LABEL"] = rank_df["REGION_RANK"].apply(
    lambda x: "Best" if x == 1 else ("Worst" if x == total_regions else f"Rank {x}")
)

rank_df["RISK_LEVEL"] = rank_df["PREDICTED_RISK_SCORE"].apply(classify_risk)

# ---------------------------------------------
# ADVANCED RISK DESCRIPTION
# ---------------------------------------------
def build_advanced_risk_info(row):
    score = row["PREDICTED_RISK_SCORE"]
    rank = int(row["REGION_RANK"])
    level = row["RISK_LEVEL"]

    factor_deltas = {
        "drought": avg_drought - row["DROUGHT"],          # positive is good
        "precipitation": row["PRECIP"] - avg_precip,      # positive is good
        "temperature": avg_temp - row["TEMP"],            # positive is good
        "land value": avg_land - row["LAND_VALUE_AVG"],   # positive is good
        "market price": row["AVG_MARKET_PRICE"] - avg_market_price  # positive is good
    }

    best_driver = max(factor_deltas, key=factor_deltas.get)
    worst_driver = min(factor_deltas, key=factor_deltas.get)

    if score < avg_risk:
        overall_position = "below the regional average risk score"
    elif score > avg_risk:
        overall_position = "above the regional average risk score"
    else:
        overall_position = "at the regional average risk score"

    crop_strengths = []
    crop_weaknesses = []

    if row["CORN_PRICE"] >= avg_corn:
        crop_strengths.append("corn")
    else:
        crop_weaknesses.append("corn")

    if row["SOYBEAN_PRICE"] >= avg_soy:
        crop_strengths.append("soybeans")
    else:
        crop_weaknesses.append("soybeans")

    if row["WHEAT_PRICE"] >= avg_wheat:
        crop_strengths.append("wheat")
    else:
        crop_weaknesses.append("wheat")

    if len(crop_strengths) == 3:
        market_text = "all three crop markets are stronger than the regional benchmark"
    elif len(crop_strengths) == 2:
        market_text = f"market support is strongest in {', '.join(crop_strengths)}"
    elif len(crop_strengths) == 1:
        market_text = f"only {crop_strengths[0]} is outperforming the regional benchmark"
    else:
        market_text = "all three crop markets are weaker than the regional benchmark"

    description = (
        f"{level} risk profile. "
        f"This region ranks {rank} out of {total_regions} and is {overall_position}. "
        f"The strongest supportive driver is {best_driver}, while the biggest pressure point is {worst_driver}. "
        f"Drought = {row['DROUGHT']:.2f}, Precipitation = {row['PRECIP']:.2f}, Temperature = {row['TEMP']:.2f}, "
        f"Land Value = {row['LAND_VALUE_AVG']:.2f}, and Average Market Price = {row['AVG_MARKET_PRICE']:.2f}. "
        f"In crop pricing terms, {market_text}."
    )

    return description

rank_df["RISK_INFO"] = rank_df.apply(build_advanced_risk_info, axis=1)

# ---------------------------------------------
# FINAL OUTPUT TABLE
# ---------------------------------------------
final_rank_table = rank_df[[
    "REGION_RANK",
    "RANK_LABEL",
    "REGION",
    "YEAR",
    "PREDICTED_RISK_SCORE",
    "RISK_LEVEL",
    "DROUGHT",
    "PRECIP",
    "TEMP",
    "LAND_VALUE_AVG",
    "AVG_MARKET_PRICE",
    "CORN_PRICE",
    "SOYBEAN_PRICE",
    "WHEAT_PRICE",
    "RISK_INFO"
]].copy()

print(f"\nREGION RANKING FOR {rank_year} (BEST TO WORST):")
print(final_rank_table)

# ---------------------------------------------
# REPORT-FRIENDLY OUTPUT TABLE
# ---------------------------------------------
report_output_table = final_rank_table[[
    "REGION_RANK",
    "REGION",
    "YEAR",
    "PREDICTED_RISK_SCORE",
    "RISK_LEVEL",
    "RANK_LABEL",
    "RISK_INFO"
]].copy()

print(f"\nREPORT OUTPUT TABLE FOR {rank_year}:")
print(report_output_table)

# ---------------------------------------------
# OPTIONAL: SAVE TO SNOWFLAKE
# ---------------------------------------------
session.write_pandas(
    final_rank_table.reset_index(drop=True),
    f"REGION_RISK_RANKING_{rank_year}",
    auto_create_table=True,
    overwrite=True
)

session.write_pandas(
    report_output_table.reset_index(drop=True),
    f"REGION_RISK_REPORT_{rank_year}",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# --------------------------------------------------
# CHOOSE DATASET
# use final_future for forecasted years or df for historical
# --------------------------------------------------
data = final_future.copy()   # change to df if needed

# Optional: choose only one year
# data = data[data["YEAR"] == 2025].copy()
# data = data[data["YEAR"] == 2026].copy()

# --------------------------------------------------
# REQUIRED COLUMNS
# --------------------------------------------------
required_cols = [
    "REGION",
    "DROUGHT",
    "PRECIP",
    "TEMP",
    "LAND_VALUE_AVG",
    "CORN_PRICE",
    "SOYBEAN_PRICE",
    "WHEAT_PRICE"
]

missing_cols = [c for c in required_cols if c not in data.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# --------------------------------------------------
# CLEAN NUMERIC COLUMNS
# --------------------------------------------------
numeric_cols = [
    "DROUGHT",
    "PRECIP",
    "TEMP",
    "LAND_VALUE_AVG",
    "CORN_PRICE",
    "SOYBEAN_PRICE",
    "WHEAT_PRICE"
]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

# --------------------------------------------------
# NORMALIZE FEATURES
# --------------------------------------------------
scaler = MinMaxScaler()
scaled = scaler.fit_transform(data[numeric_cols])

scaled_df = pd.DataFrame(
    scaled,
    columns=[f"{col}_NORM" for col in numeric_cols],
    index=data.index
)

data = pd.concat([data, scaled_df], axis=1)

# --------------------------------------------------
# CROP-SPECIFIC WEIGHTS
# --------------------------------------------------
CROP_WEIGHTS = {
    "CORN": {
        "drought": 0.35,
        "precip": 0.30,
        "temp": 0.15,
        "land": 0.10,
        "price": 0.10
    },
    "SOYBEAN": {
        "drought": 0.30,
        "precip": 0.25,
        "temp": 0.15,
        "land": 0.10,
        "price": 0.20
    },
    "WHEAT": {
        "drought": 0.25,
        "precip": 0.20,
        "temp": 0.25,
        "land": 0.10,
        "price": 0.20
    }
}

# --------------------------------------------------
# FUNCTION TO COMPUTE CROP SCORE
# Higher score = better region for that crop
# --------------------------------------------------
def compute_score(row, weights, price_col):
    return (
        weights["drought"] * (1 - row["DROUGHT_NORM"]) +
        weights["precip"] * row["PRECIP_NORM"] +
        weights["temp"] * (1 - row["TEMP_NORM"]) +
        weights["land"] * (1 - row["LAND_VALUE_AVG_NORM"]) +
        weights["price"] * row[price_col]
    ) * 100

# --------------------------------------------------
# CALCULATE CROP-SPECIFIC SCORES
# --------------------------------------------------
data["CORN_SCORE"] = data.apply(
    lambda row: compute_score(row, CROP_WEIGHTS["CORN"], "CORN_PRICE_NORM"),
    axis=1
)

data["SOYBEAN_SCORE"] = data.apply(
    lambda row: compute_score(row, CROP_WEIGHTS["SOYBEAN"], "SOYBEAN_PRICE_NORM"),
    axis=1
)

data["HARD_RED_WINTER_WHEAT_SCORE"] = data.apply(
    lambda row: compute_score(row, CROP_WEIGHTS["WHEAT"], "WHEAT_PRICE_NORM"),
    axis=1
)

score_cols = ["CORN_SCORE", "SOYBEAN_SCORE", "HARD_RED_WINTER_WHEAT_SCORE"]
for col in score_cols:
    data[col] = data[col].clip(0, 100).round(2)

# --------------------------------------------------
# AVERAGE SCORES BY REGION
# --------------------------------------------------
region_scores = (
    data.groupby("REGION", as_index=False)[score_cols]
    .mean()
)

# Also keep average raw values by region so descriptions can use them
region_features = (
    data.groupby("REGION", as_index=False)[[
        "DROUGHT",
        "PRECIP",
        "TEMP",
        "LAND_VALUE_AVG",
        "CORN_PRICE",
        "SOYBEAN_PRICE",
        "WHEAT_PRICE"
    ]]
    .mean()
)

region_scores = region_scores.merge(region_features, on="REGION", how="left")

# --------------------------------------------------
# FUNCTION TO BUILD BETTER DESCRIPTION
# --------------------------------------------------
def build_crop_ranking(region_df, score_col, crop_name, price_col, weights):
    temp = region_df.copy().sort_values(score_col, ascending=False).reset_index(drop=True)
    temp["RANK"] = temp.index + 1

    temp["RANK_LABEL"] = temp["RANK"].apply(
        lambda x: "Best" if x == 1 else ("Worst" if x == len(temp) else f"Rank {x}")
    )

    avg_score = temp[score_col].mean()
    avg_drought = temp["DROUGHT"].mean()
    avg_precip = temp["PRECIP"].mean()
    avg_temp = temp["TEMP"].mean()
    avg_land = temp["LAND_VALUE_AVG"].mean()
    avg_price = temp[price_col].mean()

    def build_info(row):
        score = row[score_col]

        if score >= avg_score + 5:
            score_text = f"{crop_name} shows a strong regional outlook here"
        elif score > avg_score:
            score_text = f"{crop_name} performs slightly above the regional average here"
        elif score <= avg_score - 5:
            score_text = f"{crop_name} faces a weaker regional outlook here"
        else:
            score_text = f"{crop_name} is close to the regional average here"

        factor_notes = []

        if row["DROUGHT"] < avg_drought:
            factor_notes.append(
                f"lower drought pressure supports the crop, which matters heavily because drought carries a weight of {weights['drought']:.2f}"
            )
        else:
            factor_notes.append(
                f"higher drought pressure works against the crop, and drought is an important factor with weight {weights['drought']:.2f}"
            )

        if row["PRECIP"] > avg_precip:
            factor_notes.append(
                f"precipitation is stronger than average, giving better moisture support with a factor weight of {weights['precip']:.2f}"
            )
        else:
            factor_notes.append(
                f"precipitation is below average, reducing moisture support under a factor weight of {weights['precip']:.2f}"
            )

        if row["TEMP"] <= avg_temp:
            factor_notes.append(
                f"temperature conditions are more moderate, which helps limit stress and is weighted at {weights['temp']:.2f}"
            )
        else:
            factor_notes.append(
                f"temperature conditions are warmer than average, which may increase crop stress under a weight of {weights['temp']:.2f}"
            )

        if row["LAND_VALUE_AVG"] < avg_land:
            factor_notes.append(
                f"land value pressure is lower, improving cost conditions with a land weight of {weights['land']:.2f}"
            )
        else:
            factor_notes.append(
                f"land value pressure is higher, which creates a less favorable cost environment under a land weight of {weights['land']:.2f}"
            )

        if row[price_col] >= avg_price:
            factor_notes.append(
                f"{crop_name.lower()} price conditions are stronger than average, which helps revenue potential and carries a market weight of {weights['price']:.2f}"
            )
        else:
            factor_notes.append(
                f"{crop_name.lower()} price conditions are weaker than average, which limits market support under a weight of {weights['price']:.2f}"
            )

        # Best/worst interpretation
        if row["RANK"] == 1:
            rank_text = f"This region ranks as the best overall option for {crop_name.lower()} in this comparison."
        elif row["RANK"] == len(temp):
            rank_text = f"This region ranks as the weakest overall option for {crop_name.lower()} in this comparison."
        else:
            rank_text = f"This region ranks {int(row['RANK'])} out of {len(temp)} for {crop_name.lower()}."

        return f"{score_text}. {rank_text} " + " ".join(factor_notes) + "."

    temp["RISK_INFO"] = temp.apply(build_info, axis=1)

    return temp[["REGION", score_col, "RANK", "RANK_LABEL", "RISK_INFO"]]

# --------------------------------------------------
# BUILD SEPARATE TABLES
# --------------------------------------------------
corn_ranking = build_crop_ranking(
    region_scores,
    "CORN_SCORE",
    "Corn",
    "CORN_PRICE",
    CROP_WEIGHTS["CORN"]
)

soybean_ranking = build_crop_ranking(
    region_scores,
    "SOYBEAN_SCORE",
    "Soybean",
    "SOYBEAN_PRICE",
    CROP_WEIGHTS["SOYBEAN"]
)

hrw_wheat_ranking = build_crop_ranking(
    region_scores,
    "HARD_RED_WINTER_WHEAT_SCORE",
    "Hard Red Winter Wheat",
    "WHEAT_PRICE",
    CROP_WEIGHTS["WHEAT"]
)

# --------------------------------------------------
# PRINT RESULTS
# --------------------------------------------------
print("\nCORN REGION RANKING")
print(corn_ranking)

print("\nSOYBEAN REGION RANKING")
print(soybean_ranking)

print("\nHARD RED WINTER WHEAT REGION RANKING")
print(hrw_wheat_ranking)

# --------------------------------------------------
# OPTIONAL: COMBINED LONG TABLE
# --------------------------------------------------
corn_out = corn_ranking.copy()
corn_out["CROP"] = "Corn"
corn_out = corn_out.rename(columns={"CORN_SCORE": "CROP_SCORE"})

soy_out = soybean_ranking.copy()
soy_out["CROP"] = "Soybean"
soy_out = soy_out.rename(columns={"SOYBEAN_SCORE": "CROP_SCORE"})

wheat_out = hrw_wheat_ranking.copy()
wheat_out["CROP"] = "Hard Red Winter Wheat"
wheat_out = wheat_out.rename(columns={"HARD_RED_WINTER_WHEAT_SCORE": "CROP_SCORE"})

all_crop_rankings = pd.concat([corn_out, soy_out, wheat_out], ignore_index=True)
all_crop_rankings = all_crop_rankings[["CROP", "REGION", "CROP_SCORE", "RANK", "RANK_LABEL", "RISK_INFO"]]

print("\nALL CROP RANKINGS COMBINED")
print(all_crop_rankings.sort_values(["CROP", "RANK"]))

# --------------------------------------------------
# OPTIONAL: SAVE TO SNOWFLAKE
# --------------------------------------------------
session.write_pandas(corn_ranking.reset_index(drop=True), "CORN_REGION_RANKING", auto_create_table=True, overwrite=True)
session.write_pandas(soybean_ranking.reset_index(drop=True), "SOYBEAN_REGION_RANKING", auto_create_table=True, overwrite=True)
session.write_pandas(hrw_wheat_ranking.reset_index(drop=True), "HRW_WHEAT_REGION_RANKING", auto_create_table=True, overwrite=True)
session.write_pandas(all_crop_rankings.reset_index(drop=True), "ALL_CROP_REGION_RANKINGS", auto_create_table=True, overwrite=True)

In [ ]:
CREATE OR REPLACE TABLE KANSAS_HISTORICAL_MARKET_RISK AS
WITH market_yearly AS (
    SELECT
        CASE
            WHEN UPPER(REGION) LIKE '%NORTHWEST%' THEN 'Northwest Kansas'
            WHEN UPPER(REGION) LIKE '%NORTHEAST%' THEN 'Northeast Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHWEST%' THEN 'Southwest Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHEAST%' THEN 'Southeast Kansas'
            WHEN UPPER(REGION) LIKE '%CENTRAL%' THEN 'Central Kansas'
            ELSE REGION
        END AS REGION,
        YEAR,
        AVG(CASE WHEN COMMODITY ILIKE '%CORN%' THEN PRICE_USD_PER_BUSHEL END) AS AVG_CORN_PRICE,
        AVG(CASE WHEN COMMODITY ILIKE '%SOY%' THEN PRICE_USD_PER_BUSHEL END) AS AVG_SOYBEAN_PRICE,
        AVG(CASE WHEN COMMODITY ILIKE '%WHEAT%' THEN PRICE_USD_PER_BUSHEL END) AS AVG_WHEAT_PRICE
    FROM MARKET_RISK_VALUE
    WHERE YEAR BETWEEN 2014 AND 2024
    GROUP BY 1, YEAR
),
factor AS (
    SELECT *,
        ROUND(
            (COALESCE(AVG_CORN_PRICE,0) +
             COALESCE(AVG_SOYBEAN_PRICE,0) +
             COALESCE(AVG_WHEAT_PRICE,0)) / 3,
            2
        ) AS MARKET_FACTOR_VALUE
    FROM market_yearly
),
risked AS (
    SELECT *,
        ROUND(
            99 * ((MARKET_FACTOR_VALUE - MIN(MARKET_FACTOR_VALUE) OVER ()) /
            NULLIF(MAX(MARKET_FACTOR_VALUE) OVER () - MIN(MARKET_FACTOR_VALUE) OVER (),0)),
        2) AS MARKET_RISK_SCORE
    FROM factor
)
SELECT *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY MARKET_RISK_SCORE DESC) AS MARKET_RISK_RANK,

    CASE
        WHEN MARKET_RISK_SCORE < 20 THEN 'Very Low'
        WHEN MARKET_RISK_SCORE < 40 THEN 'Low'
        WHEN MARKET_RISK_SCORE < 60 THEN 'Moderate'
        WHEN MARKET_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS MARKET_RISK_LEVEL,

    REGION || ' in ' || YEAR ||
    ' has a yearly market risk score of ' || MARKET_RISK_SCORE ||
    '. Corn averaged $' || ROUND(AVG_CORN_PRICE,2) ||
    ', soybeans averaged $' || ROUND(AVG_SOYBEAN_PRICE,2) ||
    ', and wheat averaged $' || ROUND(AVG_WHEAT_PRICE,2) ||
    ' per bushel. These prices reflect the overall market environment and revenue potential, where lower or more volatile prices increase financial risk exposure.'
    AS MARKET_ADVANCED_DESCRIPTION

FROM risked;

In [ ]:
SELECT *
FROM KANSAS_HISTORICAL_MARKET_RISK
ORDER BY YEAR, MARKET_RISK_RANK
LIMIT 50;

In [ ]:
CREATE OR REPLACE TABLE KANSAS_HISTORICAL_WEATHER_RISK AS
WITH weather_yearly AS (
    SELECT
        CASE
            WHEN UPPER(REGION) LIKE '%NORTHWEST%' THEN 'Northwest Kansas'
            WHEN UPPER(REGION) LIKE '%NORTHEAST%' THEN 'Northeast Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHWEST%' THEN 'Southwest Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHEAST%' THEN 'Southeast Kansas'
            WHEN UPPER(REGION) LIKE '%CENTRAL%' THEN 'Central Kansas'
            ELSE REGION
        END AS REGION,
        YEAR,
        AVG(PRECIPITATION_INCHES) AS AVG_PRECIP,
        AVG(AVERAGE_TEMPERATURE) AS AVG_TEMP,
        AVG(PALMER_DROUGHT_SEVERITY_INDEX) AS AVG_DROUGHT,
        AVG(HEATING_DEGREE_DAYS) AS AVG_HEAT,
        AVG(COOLING_DEGREE_DAYS) AS AVG_COOL
    FROM WEATHER_RISK_DATA
    WHERE YEAR BETWEEN 2014 AND 2024
    GROUP BY 1, YEAR
),
components AS (
    SELECT *,
        1 - ((AVG_PRECIP - MIN(AVG_PRECIP) OVER ()) / NULLIF(MAX(AVG_PRECIP) OVER () - MIN(AVG_PRECIP) OVER (),0)) AS PRECIP_COMPONENT,
        ((AVG_TEMP - MIN(AVG_TEMP) OVER ()) / NULLIF(MAX(AVG_TEMP) OVER () - MIN(AVG_TEMP) OVER (),0)) AS TEMP_COMPONENT,
        ((AVG_DROUGHT - MIN(AVG_DROUGHT) OVER ()) / NULLIF(MAX(AVG_DROUGHT) OVER () - MIN(AVG_DROUGHT) OVER (),0)) AS DROUGHT_COMPONENT,
        ((AVG_HEAT - MIN(AVG_HEAT) OVER ()) / NULLIF(MAX(AVG_HEAT) OVER () - MIN(AVG_HEAT) OVER (),0)) AS HEAT_COMPONENT,
        ((AVG_COOL - MIN(AVG_COOL) OVER ()) / NULLIF(MAX(AVG_COOL) OVER () - MIN(AVG_COOL) OVER (),0)) AS COOL_COMPONENT
    FROM weather_yearly
),
risked AS (
    SELECT *,
        ROUND(
            (PRECIP_COMPONENT*0.30 + DROUGHT_COMPONENT*0.30 +
             TEMP_COMPONENT*0.15 + HEAT_COMPONENT*0.125 + COOL_COMPONENT*0.125)*99,
        2) AS WEATHER_RISK_SCORE
    FROM components
)
SELECT *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY WEATHER_RISK_SCORE DESC) AS WEATHER_RISK_RANK,

    CASE
        WHEN WEATHER_RISK_SCORE < 20 THEN 'Very Low'
        WHEN WEATHER_RISK_SCORE < 40 THEN 'Low'
        WHEN WEATHER_RISK_SCORE < 60 THEN 'Moderate'
        WHEN WEATHER_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS WEATHER_RISK_LEVEL,

    REGION || ' in ' || YEAR ||
    ' has a yearly weather risk score of ' || WEATHER_RISK_SCORE ||
    '. The region averaged ' || ROUND(AVG_PRECIP,2) || ' inches of precipitation, a temperature of ' ||
    ROUND(AVG_TEMP,2) || ', and a drought index of ' || ROUND(AVG_DROUGHT,2) ||
    '. Heating and cooling demands were ' || ROUND(AVG_HEAT,2) || ' and ' || ROUND(AVG_COOL,2) ||
    '. These conditions reflect the environmental stress placed on agricultural production, with drought and precipitation variability as major drivers of risk.'
    AS WEATHER_ADVANCED_DESCRIPTION

FROM risked;

In [ ]:
SELECT *
FROM KANSAS_HISTORICAL_WEATHER_RISK
ORDER BY YEAR, WEATHER_RISK_RANK
LIMIT 50;

In [ ]:
CREATE OR REPLACE TABLE KANSAS_HISTORICAL_LAND_RISK AS
WITH land_yearly AS (
    SELECT
        CASE
            WHEN UPPER(REGION) LIKE '%NORTHWEST%' THEN 'Northwest Kansas'
            WHEN UPPER(REGION) LIKE '%NORTHEAST%' THEN 'Northeast Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHWEST%' THEN 'Southwest Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHEAST%' THEN 'Southeast Kansas'
            WHEN UPPER(REGION) LIKE '%CENTRAL%' THEN 'Central Kansas'
            ELSE REGION
        END AS REGION,
        YEAR,
        AVG(LAND_VALUE_AVG_PER_ACRE) AS AVG_LAND_VALUE,
        AVG(ALL_CROPLAND_PER_ACRE) AS AVG_CROPLAND_VALUE,
        AVG(FARM_REAL_ESTATE_PER_ACRE) AS AVG_FARM_RE_VALUE,
        AVG(CASH_RENT_DRYLAND_PER_ACRE) AS AVG_RENT_DRYLAND,
        AVG(CASH_RENT_IRRIGATED_PER_ACRE) AS AVG_RENT_IRRIGATED,
        AVG(CASH_RENT_PASTURE_PER_ACRE) AS AVG_RENT_PASTURE
    FROM LAND_VALUE_DATA
    WHERE YEAR BETWEEN 2014 AND 2024
    GROUP BY 1, YEAR
),
factor AS (
    SELECT *,
        ROUND(
            COALESCE(AVG_LAND_VALUE,0)*0.40 +
            COALESCE(AVG_CROPLAND_VALUE,0)*0.20 +
            COALESCE(AVG_FARM_RE_VALUE,0)*0.15 +
            ((COALESCE(AVG_RENT_DRYLAND,0)+COALESCE(AVG_RENT_IRRIGATED,0)+COALESCE(AVG_RENT_PASTURE,0))/3)*0.25,
        2) AS LAND_FACTOR_VALUE
    FROM land_yearly
),
risked AS (
    SELECT *,
        ROUND(
            99*((LAND_FACTOR_VALUE - MIN(LAND_FACTOR_VALUE) OVER ()) /
            NULLIF(MAX(LAND_FACTOR_VALUE) OVER () - MIN(LAND_FACTOR_VALUE) OVER (),0)),
        2) AS LAND_RISK_SCORE
    FROM factor
)
SELECT *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY LAND_RISK_SCORE DESC) AS LAND_RISK_RANK,

    CASE
        WHEN LAND_RISK_SCORE < 20 THEN 'Very Low'
        WHEN LAND_RISK_SCORE < 40 THEN 'Low'
        WHEN LAND_RISK_SCORE < 60 THEN 'Moderate'
        WHEN LAND_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS LAND_RISK_LEVEL,

    REGION || ' in ' || YEAR ||
    ' has a yearly land risk score of ' || LAND_RISK_SCORE ||
    '. Land values averaged $' || ROUND(AVG_LAND_VALUE,2) ||
    ' per acre, cropland was valued at $' || ROUND(AVG_CROPLAND_VALUE,2) ||
    ', and farm real estate at $' || ROUND(AVG_FARM_RE_VALUE,2) ||
    '. Rental rates averaged $' || ROUND(AVG_RENT_DRYLAND,2) ||
    ' (dryland), $' || ROUND(AVG_RENT_IRRIGATED,2) ||
    ' (irrigated), and $' || ROUND(AVG_RENT_PASTURE,2) ||
    '. These values represent the cost pressure and investment intensity required to operate in this region.'
    AS LAND_ADVANCED_DESCRIPTION

FROM risked;

In [ ]:
SELECT *
FROM KANSAS_HISTORICAL_LAND_RISK
ORDER BY YEAR, LAND_RISK_RANK
LIMIT 50;

In [ ]:
CREATE OR REPLACE TABLE KANSAS_HISTORICAL_TOTAL_RISK AS
WITH combined AS (
    SELECT
        m.REGION,
        m.YEAR,
        m.MARKET_RISK_SCORE,
        w.WEATHER_RISK_SCORE,
        l.LAND_RISK_SCORE,

        ROUND(
            0.40*m.MARKET_RISK_SCORE +
            0.35*w.WEATHER_RISK_SCORE +
            0.25*l.LAND_RISK_SCORE,
        2) AS TOTAL_RISK_SCORE

    FROM KANSAS_HISTORICAL_MARKET_RISK m
    JOIN KANSAS_HISTORICAL_WEATHER_RISK w ON m.REGION = w.REGION AND m.YEAR = w.YEAR
    JOIN KANSAS_HISTORICAL_LAND_RISK l ON m.REGION = l.REGION AND m.YEAR = l.YEAR
)

SELECT *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY TOTAL_RISK_SCORE DESC) AS TOTAL_RISK_RANK,

    CASE
        WHEN TOTAL_RISK_SCORE < 20 THEN 'Very Low'
        WHEN TOTAL_RISK_SCORE < 40 THEN 'Low'
        WHEN TOTAL_RISK_SCORE < 60 THEN 'Moderate'
        WHEN TOTAL_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS TOTAL_RISK_LEVEL,

    REGION || ' in ' || YEAR ||
    ' has a yearly total risk score of ' || TOTAL_RISK_SCORE ||
    ', combining market risk (' || MARKET_RISK_SCORE ||
    '), weather risk (' || WEATHER_RISK_SCORE ||
    '), and land-value risk (' || LAND_RISK_SCORE ||
    '). The score is calculated using a weighted model of 40% market, 35% weather, and 25% land risk. ' ||

    CASE
        WHEN MARKET_RISK_SCORE >= WEATHER_RISK_SCORE AND MARKET_RISK_SCORE >= LAND_RISK_SCORE
        THEN 'Market conditions are the dominant driver, indicating commodity prices and revenue volatility have the greatest impact on this region.'

        WHEN WEATHER_RISK_SCORE >= MARKET_RISK_SCORE AND WEATHER_RISK_SCORE >= LAND_RISK_SCORE
        THEN 'Weather conditions are the dominant driver, meaning precipitation variability and drought stress are the main factors affecting this region.'

        ELSE 'Land-value pressure is the dominant driver, meaning high land costs and rental rates are the primary contributors to financial risk in this region.'
    END

    AS TOTAL_ADVANCED_DESCRIPTION

FROM combined;

In [ ]:
SELECT *
FROM KANSAS_HISTORICAL_TOTAL_RISK
ORDER BY YEAR, TOTAL_RISK_RANK
LIMIT 50;

In [ ]:
CREATE OR REPLACE TABLE FUTURE_MARKET_RISK_2025_2030 AS
WITH forecast_years AS (
    SELECT 2025 AS YEAR UNION ALL
    SELECT 2026 UNION ALL
    SELECT 2027 UNION ALL
    SELECT 2028 UNION ALL
    SELECT 2029 UNION ALL
    SELECT 2030
),
market_yearly AS (
    SELECT
        REGION,
        COMMODITY,
        YEAR,
        AVG(PRICE_USD_PER_BUSHEL) AS PRICE
    FROM MARKET_RISK_VALUE
    GROUP BY REGION, COMMODITY, YEAR
),
model AS (
    SELECT
        REGION,
        COMMODITY,
        REGR_SLOPE(PRICE, YEAR) AS SLOPE,
        REGR_INTERCEPT(PRICE, YEAR) AS INTERCEPT
    FROM market_yearly
    GROUP BY REGION, COMMODITY
),
predicted AS (
    SELECT
        m.REGION,
        f.YEAR,
        m.COMMODITY,
        m.INTERCEPT + m.SLOPE * f.YEAR AS PREDICTED_PRICE
    FROM model m
    CROSS JOIN forecast_years f
),
pivoted AS (
    SELECT
        REGION,
        YEAR,
        AVG(CASE WHEN COMMODITY ILIKE '%CORN%' THEN PREDICTED_PRICE END) AS AVG_CORN_PRICE,
        AVG(CASE WHEN COMMODITY ILIKE '%SOY%' THEN PREDICTED_PRICE END) AS AVG_SOYBEAN_PRICE,
        AVG(CASE WHEN COMMODITY ILIKE '%WHEAT%' THEN PREDICTED_PRICE END) AS AVG_WHEAT_PRICE
    FROM predicted
    GROUP BY REGION, YEAR
),
factor AS (
    SELECT
        *,
        ROUND(
            (
                COALESCE(AVG_CORN_PRICE, 0) +
                COALESCE(AVG_SOYBEAN_PRICE, 0) +
                COALESCE(AVG_WHEAT_PRICE, 0)
            ) / 3,
            2
        ) AS MARKET_FACTOR_VALUE
    FROM pivoted
),
risked AS (
    SELECT
        *,
        ROUND(
            99 * (
                (MARKET_FACTOR_VALUE - MIN(MARKET_FACTOR_VALUE) OVER ()) /
                NULLIF(MAX(MARKET_FACTOR_VALUE) OVER () - MIN(MARKET_FACTOR_VALUE) OVER (), 0)
            ),
            2
        ) AS MARKET_RISK_SCORE
    FROM factor
)
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY MARKET_RISK_SCORE DESC) AS MARKET_RISK_RANK,
    CASE
        WHEN MARKET_RISK_SCORE < 20 THEN 'Very Low'
        WHEN MARKET_RISK_SCORE < 40 THEN 'Low'
        WHEN MARKET_RISK_SCORE < 60 THEN 'Moderate'
        WHEN MARKET_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS MARKET_RISK_LEVEL,
    REGION || ' in ' || YEAR ||
    ' is projected to have a yearly market risk score of ' || MARKET_RISK_SCORE ||
    '. This score is based on forecasted annual corn, soybean, and wheat price behavior.' AS MARKET_ADVANCED_DESCRIPTION
FROM risked,



In [ ]:
SELECT *
FROM FUTURE_MARKET_RISK_2025_2030
ORDER BY YEAR, REGION
LIMIT 50;

In [ ]:
CREATE OR REPLACE TABLE FUTURE_WEATHER_RISK_2025_2030 AS
WITH forecast_years AS (
    SELECT 2025 AS YEAR UNION ALL
    SELECT 2026 UNION ALL
    SELECT 2027 UNION ALL
    SELECT 2028 UNION ALL
    SELECT 2029 UNION ALL
    SELECT 2030
),
weather_yearly AS (
    SELECT
        REGION,
        YEAR,
        AVG(PRECIPITATION_INCHES) AS PRECIP,
        AVG(AVERAGE_TEMPERATURE) AS TEMP,
        AVG(PALMER_DROUGHT_SEVERITY_INDEX) AS DROUGHT,
        AVG(HEATING_DEGREE_DAYS) AS HEAT,
        AVG(COOLING_DEGREE_DAYS) AS COOL
    FROM WEATHER_RISK_DATA
    GROUP BY REGION, YEAR
),
long_data AS (
    SELECT REGION, YEAR, 'PRECIP' AS METRIC, PRECIP AS VALUE FROM weather_yearly
    UNION ALL SELECT REGION, YEAR, 'TEMP', TEMP FROM weather_yearly
    UNION ALL SELECT REGION, YEAR, 'DROUGHT', DROUGHT FROM weather_yearly
    UNION ALL SELECT REGION, YEAR, 'HEAT', HEAT FROM weather_yearly
    UNION ALL SELECT REGION, YEAR, 'COOL', COOL FROM weather_yearly
),
model AS (
    SELECT
        REGION,
        METRIC,
        REGR_SLOPE(VALUE, YEAR) AS SLOPE,
        REGR_INTERCEPT(VALUE, YEAR) AS INTERCEPT
    FROM long_data
    GROUP BY REGION, METRIC
),
predicted AS (
    SELECT
        m.REGION,
        f.YEAR,
        m.METRIC,
        m.INTERCEPT + m.SLOPE * f.YEAR AS PREDICTED_VALUE
    FROM model m
    CROSS JOIN forecast_years f
),
pivoted AS (
    SELECT
        REGION,
        YEAR,
        AVG(CASE WHEN METRIC = 'PRECIP' THEN PREDICTED_VALUE END) AS AVG_PRECIP,
        AVG(CASE WHEN METRIC = 'TEMP' THEN PREDICTED_VALUE END) AS AVG_TEMP,
        AVG(CASE WHEN METRIC = 'DROUGHT' THEN PREDICTED_VALUE END) AS AVG_DROUGHT,
        AVG(CASE WHEN METRIC = 'HEAT' THEN PREDICTED_VALUE END) AS AVG_HEAT,
        AVG(CASE WHEN METRIC = 'COOL' THEN PREDICTED_VALUE END) AS AVG_COOL
    FROM predicted
    GROUP BY REGION, YEAR
),
components AS (
    SELECT
        *,
        1 - ((AVG_PRECIP - MIN(AVG_PRECIP) OVER ()) / NULLIF(MAX(AVG_PRECIP) OVER () - MIN(AVG_PRECIP) OVER (), 0)) AS PRECIP_COMPONENT,
        ((AVG_TEMP - MIN(AVG_TEMP) OVER ()) / NULLIF(MAX(AVG_TEMP) OVER () - MIN(AVG_TEMP) OVER (), 0)) AS TEMP_COMPONENT,
        ((AVG_DROUGHT - MIN(AVG_DROUGHT) OVER ()) / NULLIF(MAX(AVG_DROUGHT) OVER () - MIN(AVG_DROUGHT) OVER (), 0)) AS DROUGHT_COMPONENT,
        ((AVG_HEAT - MIN(AVG_HEAT) OVER ()) / NULLIF(MAX(AVG_HEAT) OVER () - MIN(AVG_HEAT) OVER (), 0)) AS HEAT_COMPONENT,
        ((AVG_COOL - MIN(AVG_COOL) OVER ()) / NULLIF(MAX(AVG_COOL) OVER () - MIN(AVG_COOL) OVER (), 0)) AS COOL_COMPONENT
    FROM pivoted
),
risked AS (
    SELECT
        *,
        ROUND(
            (
                COALESCE(PRECIP_COMPONENT, 0) * 0.30 +
                COALESCE(DROUGHT_COMPONENT, 0) * 0.30 +
                COALESCE(TEMP_COMPONENT, 0) * 0.15 +
                COALESCE(HEAT_COMPONENT, 0) * 0.125 +
                COALESCE(COOL_COMPONENT, 0) * 0.125
            ) * 99,
            2
        ) AS WEATHER_RISK_SCORE
    FROM components
)
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY WEATHER_RISK_SCORE DESC) AS WEATHER_RISK_RANK,
    CASE
        WHEN WEATHER_RISK_SCORE < 20 THEN 'Very Low'
        WHEN WEATHER_RISK_SCORE < 40 THEN 'Low'
        WHEN WEATHER_RISK_SCORE < 60 THEN 'Moderate'
        WHEN WEATHER_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS WEATHER_RISK_LEVEL,
    REGION || ' in ' || YEAR ||
    ' is projected to have a yearly weather risk score of ' || WEATHER_RISK_SCORE ||
    '. This score reflects projected annual precipitation, drought pressure, temperature, heating degree days, and cooling degree days.' AS WEATHER_ADVANCED_DESCRIPTION
FROM risked;

In [ ]:
SELECT *
FROM FUTURE_WEATHER_RISK_2025_2030
ORDER BY YEAR, REGION
LIMIT 50;

In [ ]:
CREATE OR REPLACE TABLE FUTURE_LAND_RISK_2025_2030 AS
WITH standardized AS (
    SELECT
        CASE
            WHEN LOWER(TRIM(REGION)) = 'northwest' THEN 'Northwest Kansas'
            WHEN LOWER(TRIM(REGION)) = 'northeast' THEN 'Northeast Kansas'
            WHEN LOWER(TRIM(REGION)) = 'southwest' THEN 'Southwest Kansas'
            WHEN LOWER(TRIM(REGION)) = 'southeast' THEN 'Southeast Kansas'
            WHEN LOWER(TRIM(REGION)) = 'central' THEN 'Central Kansas'
            ELSE INITCAP(TRIM(REGION))
        END AS REGION,

        YEAR,

        LAND_VALUE,
        CROPLAND_VALUE,
        FARM_RE_VALUE,
        RENT_DRYLAND,
        RENT_IRRIGATED,
        RENT_PASTURE

    FROM FUTURE_LAND_RISK_2025_2030
),

factor AS (
    SELECT *,
        ROUND(
            COALESCE(LAND_VALUE, 0) * 0.40 +
            COALESCE(CROPLAND_VALUE, 0) * 0.20 +
            COALESCE(FARM_RE_VALUE, 0) * 0.15 +
            (
                (
                    COALESCE(RENT_DRYLAND, 0) +
                    COALESCE(RENT_IRRIGATED, 0) +
                    COALESCE(RENT_PASTURE, 0)
                ) / 3
            ) * 0.25,
        2) AS LAND_FACTOR_VALUE
    FROM standardized
),

risked AS (
    SELECT *,
        ROUND(
            99 * (
                (LAND_FACTOR_VALUE - MIN(LAND_FACTOR_VALUE) OVER ()) /
                NULLIF(MAX(LAND_FACTOR_VALUE) OVER () - MIN(LAND_FACTOR_VALUE) OVER (), 0)
            ),
        2) AS LAND_RISK_SCORE
    FROM factor
),

ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY YEAR
            ORDER BY LAND_RISK_SCORE DESC
        ) AS LAND_RISK_RANK
    FROM risked
)

SELECT
    REGION,
    YEAR,

    LAND_VALUE,
    CROPLAND_VALUE,
    FARM_RE_VALUE,
    RENT_DRYLAND,
    RENT_IRRIGATED,
    RENT_PASTURE,

    LAND_FACTOR_VALUE,
    LAND_RISK_SCORE,
    LAND_RISK_RANK,

    CASE
        WHEN LAND_RISK_SCORE < 20 THEN 'Very Low'
        WHEN LAND_RISK_SCORE < 40 THEN 'Low'
        WHEN LAND_RISK_SCORE < 60 THEN 'Moderate'
        WHEN LAND_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS LAND_RISK_LEVEL,

    REGION || ' in ' || YEAR ||
    ' is projected to have a yearly land-value risk score of ' || LAND_RISK_SCORE ||
    '. Forecasted land value is $' || ROUND(LAND_VALUE, 2) ||
    ' per acre, cropland value is $' || ROUND(CROPLAND_VALUE, 2) ||
    ', and farm real estate value is $' || ROUND(FARM_RE_VALUE, 2) ||
    '. Rental rates are projected at $' || ROUND(RENT_DRYLAND, 2) ||
    ' (dryland), $' || ROUND(RENT_IRRIGATED, 2) ||
    ' (irrigated), and $' || ROUND(RENT_PASTURE, 2) ||
    '. These values indicate future land cost pressure and capital requirements for producers in the region.'
    AS LAND_ADVANCED_DESCRIPTION

FROM ranked;

In [ ]:
SELECT *
FROM FUTURE_LAND_RISK_2025_2030
ORDER BY YEAR, REGION
LIMIT 50;

In [ ]:
CREATE OR REPLACE TABLE KANSAS_FUTURE_TOTAL_RISK_2025_2030 AS
WITH market_clean AS (
    SELECT
        CASE
            WHEN UPPER(REGION) LIKE '%NORTHWEST%' THEN 'Northwest Kansas'
            WHEN UPPER(REGION) LIKE '%NORTHEAST%' THEN 'Northeast Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHWEST%' THEN 'Southwest Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHEAST%' THEN 'Southeast Kansas'
            WHEN UPPER(REGION) LIKE '%CENTRAL%' THEN 'Central Kansas'
            ELSE INITCAP(TRIM(REGION))
        END AS REGION,
        YEAR,
        AVG(MARKET_RISK_SCORE) AS MARKET_RISK_SCORE
    FROM FUTURE_MARKET_RISK_2025_2030
    GROUP BY 1, YEAR
),

weather_clean AS (
    SELECT
        CASE
            WHEN UPPER(REGION) LIKE '%NORTHWEST%' THEN 'Northwest Kansas'
            WHEN UPPER(REGION) LIKE '%NORTHEAST%' THEN 'Northeast Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHWEST%' THEN 'Southwest Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHEAST%' THEN 'Southeast Kansas'
            WHEN UPPER(REGION) LIKE '%CENTRAL%' THEN 'Central Kansas'
            ELSE INITCAP(TRIM(REGION))
        END AS REGION,
        YEAR,
        AVG(WEATHER_RISK_SCORE) AS WEATHER_RISK_SCORE
    FROM FUTURE_WEATHER_RISK_2025_2030
    GROUP BY 1, YEAR
),

land_clean AS (
    SELECT
        CASE
            WHEN UPPER(REGION) LIKE '%NORTHWEST%' THEN 'Northwest Kansas'
            WHEN UPPER(REGION) LIKE '%NORTHEAST%' THEN 'Northeast Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHWEST%' THEN 'Southwest Kansas'
            WHEN UPPER(REGION) LIKE '%SOUTHEAST%' THEN 'Southeast Kansas'
            WHEN UPPER(REGION) LIKE '%CENTRAL%' THEN 'Central Kansas'
            ELSE INITCAP(TRIM(REGION))
        END AS REGION,
        YEAR,
        AVG(LAND_RISK_SCORE) AS LAND_RISK_SCORE
    FROM FUTURE_LAND_RISK_2025_2030
    GROUP BY 1, YEAR
),

combined AS (
    SELECT
        m.REGION,
        m.YEAR,
        m.MARKET_RISK_SCORE,
        w.WEATHER_RISK_SCORE,
        l.LAND_RISK_SCORE,
        ROUND(
            0.40 * m.MARKET_RISK_SCORE +
            0.35 * w.WEATHER_RISK_SCORE +
            0.25 * l.LAND_RISK_SCORE,
            2
        ) AS TOTAL_RISK_SCORE
    FROM market_clean m
    JOIN weather_clean w
        ON m.REGION = w.REGION
       AND m.YEAR = w.YEAR
    JOIN land_clean l
        ON m.REGION = l.REGION
       AND m.YEAR = l.YEAR
),

ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY YEAR
            ORDER BY TOTAL_RISK_SCORE DESC
        ) AS TOTAL_RISK_RANK
    FROM combined
)

SELECT
    REGION,
    YEAR,
    MARKET_RISK_SCORE,
    WEATHER_RISK_SCORE,
    LAND_RISK_SCORE,
    TOTAL_RISK_SCORE,
    TOTAL_RISK_RANK,

    CASE
        WHEN TOTAL_RISK_SCORE < 20 THEN 'Very Low'
        WHEN TOTAL_RISK_SCORE < 40 THEN 'Low'
        WHEN TOTAL_RISK_SCORE < 60 THEN 'Moderate'
        WHEN TOTAL_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS TOTAL_RISK_LEVEL,

    REGION || ' in ' || YEAR ||
    ' is projected to have a yearly total risk score of ' || TOTAL_RISK_SCORE ||
    '. This score combines market risk (' || MARKET_RISK_SCORE ||
    '), weather risk (' || WEATHER_RISK_SCORE ||
    '), and land-value risk (' || LAND_RISK_SCORE ||
    ') using the weighted formula: 40% market, 35% weather, and 25% land.' 
    AS TOTAL_ADVANCED_DESCRIPTION

FROM ranked;

In [ ]:
SELECT *
FROM KANSAS_FUTURE_TOTAL_RISK_2025_2030
ORDER BY YEAR, TOTAL_RISK_RANK;